In [1]:
import sys
import subprocess

def install_package(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "--quiet"])

required_packages = [
    "torch",
    "torchaudio",
    "transformers",
    "datasets",
    "evaluate",
    "jiwer",
    "librosa",
    "numpy"
]

for package in required_packages:
    install_package(package)

In [ ]:
import os
import torch
import torchaudio
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, TrainingArguments, Trainer
import evaluate
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)
model.to(device)

# Load metric for evaluation
wer_metric = evaluate.load("wer")

In [ ]:
def prepare_dataset(batch):
    # Load and resample audio data
    audio = batch["audio"]
    
    # Get input features from processor
    input_features = processor(
        audio["array"],
        sampling_rate=audio["sampling_rate"],
        return_tensors="pt"
    ).input_features[0]
    
    # Get labels from processor
    batch["labels"] = processor(text=batch["sentence"]).input_ids
    batch["input_features"] = input_features
    
    return batch

# Load dataset (using a small subset for demonstration)
dataset = load_dataset(
    "mozilla-foundation/common_voice_13_0",
    "en",
    split={
        "train": "train[:1000]",
        "test": "test[:100]"
    }
)

# Cast audio data to correct format
dataset = dataset.cast_column("audio", Audio(sampling_rate=16000))

# Preprocess datasets
processed_dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset["train"].column_names,
    num_proc=4
)

def compute_metrics(pred):
    pred_ids = np.argmax(pred.predictions, axis=-1)
    label_ids = pred.label_ids
    
    # Replace -100 with pad token id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    
    # Convert ids to strings
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    
    # Compute WER
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    
    return {"wer": wer}

In [ ]:
training_args = TrainingArguments(
    output_dir="whisper-fine-tuned",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    warmup_steps=500,
    max_steps=4000,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    logging_steps=25,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["test"],
    compute_metrics=compute_metrics,
    tokenizer=processor.feature_extractor,
)

trainer.train()

# Save the fine-tuned model
trainer.save_model("whisper-fine-tuned-final")

# Test the model
def transcribe_audio(audio_path):
    # Load and preprocess audio
    audio, sr = torchaudio.load(audio_path)
    if sr != 16000:
        audio = torchaudio.functional.resample(audio, sr, 16000)
    
    # Convert to mono if stereo
    if audio.shape[0] > 1:
        audio = torch.mean(audio, dim=0, keepdim=True)
    
    # Get input features
    input_features = processor(
        audio.squeeze().numpy(),
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(device)
    
    # Generate transcription
    predicted_ids = model.generate(input_features)
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    
    return transcription

# Example usage (uncomment and provide path to test):
# audio_path = "path/to/your/audio.wav"
# transcription = transcribe_audio(audio_path)
# print(f"Transcription: {transcription}")

In [ ]:
import matplotlib.pyplot as plt

def plot_training_metrics(trainer):
    train_metrics = trainer.state.log_history
    
    train_losses = []
    eval_losses = []
    wer_scores = []
    steps = []
    
    for metric in train_metrics:
        if 'loss' in metric:
            train_losses.append(metric['loss'])
            steps.append(metric['step'])
        if 'eval_loss' in metric:
            eval_losses.append(metric['eval_loss'])
            wer_scores.append(metric['eval_wer'])
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(steps, train_losses, label='Training Loss')
    if eval_losses:
        eval_steps = [step for step in steps if step % training_args.eval_steps == 0]
        plt.plot(eval_steps, eval_losses, label='Validation Loss')
    plt.xlabel('Training Steps')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    if wer_scores:
        plt.plot(eval_steps, wer_scores, 'r-o')
        plt.xlabel('Training Steps')
        plt.ylabel('WER')
        plt.title('Word Error Rate on Validation Set')
    
    plt.tight_layout()
    plt.show()

# Plot training progress
plot_training_metrics(trainer)

In [ ]:
def evaluate_model(audio_files):
    model.eval()
    results = []
    
    for audio_path in audio_files:
        try:
            transcription = transcribe_audio(audio_path)
            
            result = {
                'file': os.path.basename(audio_path),
                'transcription': transcription
            }
            results.append(result)
            
            print(f"\nFile: {result['file']}")
            print(f"Transcription: {result['transcription']}")
            
        except Exception as e:
            print(f"Error processing {audio_path}: {str(e)}")
    
    return results

# Example usage (uncomment and modify paths to test):
"""
test_files = [
    "path/to/audio1.wav",
    "path/to/audio2.wav",
    "path/to/audio3.wav"
]
results = evaluate_model(test_files)
"""

# How to Use This Notebook

1. Run all cells in sequence
2. The code will:
   - Install required packages
   - Download and prepare the Whisper model
   - Load and preprocess the Common Voice dataset
   - Train the model with progress tracking
   - Save the fine-tuned model
   
3. For custom dataset:
   - Place your audio files (.wav format) in a directory
   - Create a CSV file with columns: filename,text (optional)
   - Use the `process_custom_dataset()` function
   - Fine-tune the model on your data
   
4. For inference:
   - Use `transcribe_audio()` for single file transcription
   - Use `evaluate_model()` for batch transcription
   
5. Training progress visualization:
   - Loss curves and WER scores will be plotted
   - Monitor training progress in real-time

Note: Adjust batch size and other training parameters based on your GPU memory.